In [1]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

INPUT_CSV = 'earnings_calls.csv'
OUTPUT_FILE = 'call_level_sentiment.csv'
TEXT_COLUMN = 'transcript'
MODEL_NAME = 'yiyanghkust/finbert-tone'
WINDOW_SIZE = 300  # words per window
MAX_CALLS = None  # Set to a small integer while testing, e.g. 25.

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()


ModuleNotFoundError: No module named 'transformers'

In [ ]:
def analyze_call_text(full_text, call_id):
    """Analyze one earnings call with fixed-size word windows."""
    words = str(full_text).split()
    if not words:
        return None

    windows = [
        ' '.join(words[i:i + WINDOW_SIZE])
        for i in range(0, len(words), WINDOW_SIZE)
    ]

    window_scores = []
    for window in windows:
        inputs = tokenizer(
            window,
            return_tensors='pt',
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=1)

        window_scores.append({
            'neutral': probs[0][0].item(),
            'positive': probs[0][1].item(),
            'negative': probs[0][2].item()
        })

    avg = pd.DataFrame(window_scores).mean()
    return {
        'call_id': call_id,
        'neutral': avg['neutral'],
        'positive': avg['positive'],
        'negative': avg['negative'],
        'diff': avg['positive'] - avg['negative'],
        'full_text': full_text
    }

calls = pd.read_csv(INPUT_CSV).dropna(subset=[TEXT_COLUMN]).copy()
calls[TEXT_COLUMN] = calls[TEXT_COLUMN].astype(str)
if MAX_CALLS is not None:
    calls = calls.head(MAX_CALLS)

final_results = []
for call_id, full_text in tqdm(
    enumerate(calls[TEXT_COLUMN]),
    total=len(calls),
    desc='Analyzing calls'
):
    result = analyze_call_text(full_text, call_id)
    if result:
        final_results.append(result)

df_final = pd.DataFrame(final_results)
df_final.to_csv(OUTPUT_FILE, index=False)

print(f'Processed {len(df_final):,} calls')
print(f'Saved results to {OUTPUT_FILE}')
df_final.head(10)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ============================================
# 1. ENHANCED HISTOGRAM
# ============================================
fig, ax = plt.subplots(figsize=(10, 6))

# Create histogram
df_final['diff'].hist(
    bins=20 if len(df_final) > 20 else 5,  # Adjust bins based on data
    color='skyblue', 
    edgecolor='black',
    alpha=0.7,
    ax=ax
)

# Add vertical line at zero (neutral sentiment)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral (0)')

# Add mean line
mean_sentiment = df_final['diff'].mean()
ax.axvline(x=mean_sentiment, color='green', linestyle='--', linewidth=2, 
           label=f'Mean ({mean_sentiment:.3f})')

# Labels and title
ax.set_title('Distribution of Net Sentiment (Positive - Negative)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Score (Positive - Negative)', fontsize=12)
ax.set_ylabel('Number of Calls', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("net_sentiment_hist.png", dpi=300, bbox_inches="tight")

plt.show()

# ============================================
# 2. SUMMARY STATISTICS
# ============================================
print("\n=== Sentiment Statistics ===")
print(f"Total calls analyzed: {len(df_final)}")
print(f"\nNet Sentiment (diff):")
print(f"  Mean:   {df_final['diff'].mean():.3f}")
print(f"  Median: {df_final['diff'].median():.3f}")
print(f"  Std:    {df_final['diff'].std():.3f}")
print(f"  Min:    {df_final['diff'].min():.3f}")
print(f"  Max:    {df_final['diff'].max():.3f}")

# Count positive vs negative calls
positive_calls = (df_final['diff'] > 0).sum()
negative_calls = (df_final['diff'] < 0).sum()
neutral_calls = (df_final['diff'] == 0).sum()

print(f"\nCall Distribution:")
print(f"  Positive calls: {positive_calls} ({positive_calls/len(df_final)*100:.1f}%)")
print(f"  Negative calls: {negative_calls} ({negative_calls/len(df_final)*100:.1f}%)")
print(f"  Neutral calls:  {neutral_calls} ({neutral_calls/len(df_final)*100:.1f}%)")

# ============================================
# 3. MULTIPLE VISUALIZATIONS
# ============================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Histogram of diff
axes[0, 0].hist(df_final['diff'], bins=20, color='skyblue', 
                edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Net Sentiment Distribution (Positive - Negative)')
axes[0, 0].set_xlabel('Sentiment Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# 3.2 Box plot of all sentiment types
sentiment_data = df_final[['positive', 'negative', 'neutral']].values
axes[0, 1].boxplot(sentiment_data, labels=['Positive', 'Negative', 'Neutral'])
axes[0, 1].set_title('Sentiment Score Distributions by Type')
axes[0, 1].set_ylabel('Score')
axes[0, 1].grid(True, alpha=0.3)

# 3.3 Sentiment over call sequence
axes[1, 0].plot(df_final['call_id'], df_final['diff'], 
                marker='o', linestyle='-', linewidth=2, markersize=6)
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Sentiment Trend Across Calls')
axes[1, 0].set_xlabel('Call ID')
axes[1, 0].set_ylabel('Net Sentiment')
axes[1, 0].grid(True, alpha=0.3)

# 3.4 Stacked bar chart
df_final_subset = df_final.head(10)  # Show first 10 calls
x = df_final_subset['call_id']
axes[1, 1].bar(x, df_final_subset['positive'], label='Positive', 
               color='green', alpha=0.7)
axes[1, 1].bar(x, df_final_subset['neutral'], 
               bottom=df_final_subset['positive'],
               label='Neutral', color='gray', alpha=0.7)
axes[1, 1].bar(x, df_final_subset['negative'], 
               bottom=df_final_subset['positive'] + df_final_subset['neutral'],
               label='Negative', color='red', alpha=0.7)
axes[1, 1].set_title('Sentiment Composition (First 10 Calls)')
axes[1, 1].set_xlabel('Call ID')
axes[1, 1].set_ylabel('Sentiment Scores')
axes[1, 1].legend()
axes[1, 1].set_xticks(x)

plt.tight_layout()
# ✅ ADD THIS
fig.savefig("sentiment_overview.png", dpi=300, bbox_inches="tight")


plt.show()

# ============================================
# 4. PIE CHART - OVERALL SENTIMENT
# ============================================
fig, ax = plt.subplots(figsize=(8, 8))

# Calculate average sentiment across all calls
avg_sentiments = df_final[['positive', 'negative', 'neutral']].mean()

colors = ['#2ecc71', '#e74c3c', '#95a5a6']
explode = (0.1, 0, 0)  # Explode the positive slice

ax.pie(avg_sentiments, labels=['Positive', 'Negative', 'Neutral'],
       autopct='%1.1f%%', startangle=90, colors=colors, explode=explode,
       shadow=True)
ax.set_title('Average Sentiment Distribution Across All Calls', 
             fontsize=14, fontweight='bold')

# ✅ ADD THIS
fig.savefig("avg_sentiment_pie.png", dpi=300, bbox_inches="tight")

plt.show()

# ============================================
# 5. TOP/BOTTOM CALLS
# ============================================
print("\n=== Top 5 Most Positive Calls ===")
print(df_final.nlargest(5, 'diff')[['call_id', 'positive', 'negative', 'diff']])

print("\n=== Top 5 Most Negative Calls ===")
print(df_final.nsmallest(5, 'diff')[['call_id', 'positive', 'negative', 'diff']])
